# 09 · Tuning honestly, and telling real differences from luck

**Goal:** (1) tune a model with rolling-origin validation and recognise a badly designed search grid; (2) judge whether a difference between two models is real, with a paired block bootstrap.

The second half matters most. With only 26 test weeks, a difference of "5.78 vs 5.76" means nothing, and "6% better than naive" can be consistent with zero.

In [1]:
from tutorial_utils import *
style()
rl = load_panel("realistic")
h = 2
feats = build_features(rl, h)
cols = columns_for(ABLATION_SETS["E4 +cross-market"])

## Part 1 - tuning with rolling origins
**Hyper-parameters** (learning rate, tree depth, number of boosting rounds…) are chosen by trying several settings and keeping the best. "Best" must be judged on data the model was **not** trained on, and *never* on the test window.
We use the five expanding rolling-origin folds from notebook 06 and judge each setting by its **pooled** MAE across all folds.

In [2]:
def rolling_folds(first_train_end=65, step=13, block=13, last=VAL_END):
    out, te = [], first_train_end
    while te + block <= last:
        out.append((te, te + 1, te + block)); te += step
    return out
folds = rolling_folds()

def cv_mae(params, kind="hist_gb"):
    errs = []
    for tr_end, lo, hi in folds:
        p = fit_predict(feats, cols, kind, h, tr_end, lo, eval_hi(h, hi), **params)
        d = p[p.scored]; errs.append((d.y_obs - d.y_pred).abs().values)
    return float(np.concatenate(errs).mean())

import itertools, time
grid = {"learning_rate": [0.01, 0.03, 0.08], "max_iter": [100, 200, 300], "max_depth": [3, 6]}
t0 = time.time()
rows = [dict(zip(grid, combo), cv_mae=cv_mae(dict(zip(grid, combo)))) for combo in itertools.product(*grid.values())]
tuned = pd.DataFrame(rows).sort_values("cv_mae").reset_index(drop=True)
print(f"{len(rows)} settings x {len(folds)} folds = {len(rows) * len(folds)} model fits in {time.time() - t0:.0f}s")
tuned.head(8).round(3)

18 settings x 5 folds = 90 model fits in 20s


,learning_rate,max_iter,max_depth,cv_mae
0,0.03,300,3,3.739
1,0.08,100,3,3.749
2,0.03,200,3,3.762
3,0.08,200,3,3.765
4,0.03,200,6,3.794
5,0.03,100,6,3.805
6,0.08,300,3,3.811
7,0.03,300,6,3.812


### Is the grid wide enough? The edge-of-grid check
If the best value of a parameter is the **smallest or largest value you offered**, the true optimum may lie outside your grid - you have found the best of a bad menu.
This happened in the real pipeline: the first Ridge grid ended at alpha = 1000 and the winner was 1000; the first decision-tree grid started at depth 3 and the winner was 3; gradient boosting chose the smallest learning rate and
smallest number of rounds on its list. The grids were widened and the searches rerun. Here is the check, in code:

In [3]:
best = tuned.iloc[0]
for name, values in grid.items():
    at_edge = best[name] in (min(values), max(values))
    pos = "LOWEST" if best[name] == min(values) else "HIGHEST" if best[name] == max(values) else "interior"
    print(f"{name:14s} best = {best[name]:<6}  grid = {values}   -> {pos}{'  <-- consider widening the grid' if at_edge else ''}")

learning_rate  best = 0.03    grid = [0.01, 0.03, 0.08]   -> interior
max_iter       best = 300.0   grid = [100, 200, 300]   -> HIGHEST  <-- consider widening the grid
max_depth      best = 3.0     grid = [3, 6]   -> LOWEST  <-- consider widening the grid


Parameters at an edge are not always a problem (a depth of 3 may be the lowest sensible value), but each one deserves a deliberate decision: *widen it, or justify the limit*.
Also look at how **flat** the top of the table is: if the best five settings differ by 1-2%, the choice among them barely matters and spending more effort on tuning will not help.

### How much does tuning buy on data it has never seen?
Compare the tuned parameters with a plain default on the validation and the test window:

In [4]:
default = dict(learning_rate=0.05, max_iter=150, max_depth=3)
tuned_params = {k: (int(best[k]) if k != "learning_rate" else float(best[k])) for k in grid}
out = []
for name, params in [("default", default), ("tuned (rolling-origin)", tuned_params)]:
    for part, (tr_end, lo, hi) in {"validation": (TRAIN_END, TRAIN_END + 1, VAL_END), "test": (VAL_END, VAL_END + 1, TEST_END)}.items():
        p = fit_predict(feats, cols, "hist_gb", h, tr_end, lo, eval_hi(h, hi), **params)
        out.append(dict(settings=name, partition=part, MAE=score(p), naive_MAE=score(naive_pred(feats, lo, eval_hi(h, hi)))))
tt = pd.DataFrame(out); tt["skill %"] = (1 - tt.MAE / tt.naive_MAE) * 100
tt.round(2)

,settings,partition,MAE,naive_MAE,skill %
0,default,validation,3.89,4.86,19.97
1,default,test,5.83,5.88,0.89
2,tuned (rolling-origin),validation,3.89,4.86,19.90
3,tuned (rolling-origin),test,5.74,5.88,2.45


In this run tuning changes validation skill by less than 0.1 point (19.9% vs 20.0%) and moves test skill from about 0.9% to 2.5% - small either way, and (as Part 2 shows) well inside the statistical noise. The gain from tuning is usually small compared with the gain
from *information* (notebook 08): keep tuning budgets modest. And no amount of tuning on weeks ≤ 130 rescues a model from a validation-to-test shift.

## Part 2 - is the difference real?
We have 26 test weeks. If two models differ by a few percent, would a different sample of weeks reverse the ordering? The **paired block bootstrap** answers this:
* *Paired* - both models are scored on exactly the same rows, so we look at the **difference** of their errors.
* *Block* - we resample whole **origin weeks**, not individual rows, because the 20 series in a week are not independent (they share the week's shocks).
* Repeat 2,000 times; the spread of the average difference gives a confidence interval.

In [5]:
def paired_block_bootstrap(a, b, n_boot=2000, seed=0):
    keys = ["series", "origin_week"]
    m = a[keys + ["y_obs", "y_pred", "scored"]].merge(b[keys + ["y_pred"]], on=keys, suffixes=("_a", "_b"))
    m = m[m.scored]
    m["d"] = (m.y_obs - m.y_pred_a).abs() - (m.y_obs - m.y_pred_b).abs()            # negative = model A is better
    g = m.groupby("origin_week").d.agg(["sum", "count"])
    s, c = g["sum"].values, g["count"].values
    idx = np.random.default_rng(seed).integers(0, len(s), size=(n_boot, len(s)))
    boots = s[idx].sum(1) / c[idx].sum(1)
    return dict(mean_diff=s.sum() / c.sum(), ci_low=np.percentile(boots, 2.5), ci_high=np.percentile(boots, 97.5), prob_A_better=(boots < 0).mean(), weeks=len(s), rows=int(c.sum()))

preds = {}
for hh in HORIZONS:
    f = build_features(rl, hh)
    lo, hi = VAL_END + 1, eval_hi(hh, TEST_END)
    preds[hh] = {"naive": naive_pred(f, lo, hi),
                 "ridge": fit_predict(f, cols, "ridge", hh, VAL_END, lo, hi, alpha=100.0),
                 "random_forest": fit_predict(f, cols, "random_forest", hh, VAL_END, lo, hi),
                 "hist_gb": fit_predict(f, cols, "hist_gb", hh, VAL_END, lo, hi, **tuned_params)}
rows = []
for hh in HORIZONS:
    for a, b in [("hist_gb", "naive"), ("random_forest", "naive"), ("random_forest", "hist_gb"), ("random_forest", "ridge")]:
        r = paired_block_bootstrap(preds[hh][a], preds[hh][b]); r.update(horizon=hh, comparison=f"{a} vs {b}"); rows.append(r)
bt = pd.DataFrame(rows)
bt["significant"] = np.where((bt.ci_high < 0) | (bt.ci_low > 0), "yes", "no")
bt[["comparison", "horizon", "mean_diff", "ci_low", "ci_high", "prob_A_better", "significant"]].round(3)

,comparison,horizon,mean_diff,ci_low,ci_high,prob_A_better,significant
0,hist_gb vs naive,1,-0.126,-0.353,0.100,0.858,no
1,random_forest vs naive,1,-0.063,-0.277,0.149,0.712,no
2,random_forest vs hist_gb,1,0.063,-0.034,0.166,0.114,no
3,random_forest vs ridge,1,0.300,0.094,0.518,0.002,yes
4,hist_gb vs naive,2,-0.144,-0.619,0.349,0.706,no
5,random_forest vs naive,2,-0.158,-0.623,0.327,0.736,no
6,random_forest vs hist_gb,2,-0.013,-0.150,0.127,0.584,no
7,random_forest vs ridge,2,0.473,0.263,0.688,0.000,yes
8,hist_gb vs naive,4,-0.204,-1.193,0.849,0.664,no
9,random_forest vs naive,4,-0.157,-1.076,0.782,0.630,no


**How to read a row.** `mean_diff` is (model A's MAE − model B's MAE) in KES/kg: negative means A is better. If the 95% interval `[ci_low, ci_high]` contains 0, the data cannot distinguish the models; `prob_A_better` is the share of resamples in which A won.
What you should see in this untuned setup: neither gradient boosting nor the forest is **distinguishable from naive** at any horizon (intervals include 0); the forest and boosting are not distinguishable from each other; but **Ridge beats the random forest significantly at all three horizons**
(positive `mean_diff` for "random_forest vs ridge"). Whether a linear or a tree model wins depends on tuning and horizon - the pipeline's tuned models gave a different ordering (see below) - which is exactly why we test instead of eyeballing.

### The answer depends on *which weeks* you look at
The same comparison (random forest vs naive, horizon 2), using only the first 8, the first 14, or all 24 test origin weeks that exist for horizon 2:

In [6]:
a, b = preds[2]["random_forest"], preds[2]["naive"]
rows = []
for n_weeks in (8, 14, 99):
    keep = sorted(a.origin_week.unique())[:n_weeks]
    r = paired_block_bootstrap(a[a.origin_week.isin(keep)], b[b.origin_week.isin(keep)]); r["test weeks used"] = len(keep); rows.append(r)
pd.DataFrame(rows)[["test weeks used", "mean_diff", "ci_low", "ci_high", "prob_A_better"]].round(3)

,test weeks used,mean_diff,ci_low,ci_high,prob_A_better
0,8,-0.559,-0.792,-0.357,1.000
1,14,-0.222,-0.571,0.169,0.872
2,24,-0.158,-0.623,0.327,0.736


This is not what a textbook on sample size would predict, and it is more instructive. **In the first 8 test weeks - before the new shocks arrive - the forest clearly beats naive** (interval entirely below zero).
As the shock-heavy later weeks enter, the advantage shrinks and the interval *widens* to include zero: later weeks are far more volatile, so interval width reflects *dispersion*, not just the number of weeks.
A short, calm window gives a confident-looking and unrepresentative answer. Two habits follow: **evaluate over windows that include the hard periods**, and **report uncertainty with every comparison** - a table of MAEs alone invites over-reading.

### Compare with the pipeline's tests
The pipeline ran the same kind of test on its tuned models (`results/paired_bootstrap_tests_test.csv`):

In [7]:
pb = pd.read_csv(RESULTS / "paired_bootstrap_tests_test.csv")
pb["significant"] = np.where((pb.ci_high < 0) | (pb.ci_low > 0), "yes", "no")
sel = pb[pb.comparison.isin(["best ML vs naive", "Random Forest vs Ridge (E4)", "HistGB vs Random Forest (E4)", "Random Forest: E4 vs E0 (feature enrichment)"])]
sel[["comparison", "horizon", "mean_diff", "ci_low", "ci_high", "significant"]].round(3)

,comparison,horizon,mean_diff,ci_low,ci_high,significant
0,best ML vs naive,1,-0.256,-0.498,-0.026,yes
1,best ML vs naive,2,-0.204,-0.769,0.400,no
2,best ML vs naive,4,-0.684,-1.581,0.274,no
9,Random Forest: E4 vs E0 (feature enrichment),1,-0.552,-0.825,-0.312,yes
10,Random Forest: E4 vs E0 (feature enrichment),2,-0.655,-0.924,-0.407,yes
11,Random Forest: E4 vs E0 (feature enrichment),4,-0.574,-0.921,-0.241,yes
18,Random Forest vs Ridge (E4),1,0.098,-0.104,0.308,no
19,Random Forest vs Ridge (E4),2,0.439,0.164,0.736,yes
20,Random Forest vs Ridge (E4),4,-0.566,-0.888,-0.232,yes
24,HistGB vs Random Forest (E4),1,-0.021,-0.148,0.111,no


Note what the pipeline found: *enrichment* (E4 vs E0) is significant at every horizon, the model-vs-naive gain only at the 1-week horizon, and **the choice between random forest and gradient boosting is not significant anywhere**.
This is why the final report recommends a *family of models* rather than naming a single winner.

## Statistical vs practical significance
* A difference can be **statistically** clear but too small to matter (a 0.3% gain), or **practically** large but statistically uncertain (a 6% gain over 26 weeks). Report both the size and the interval.
* We ran several comparisons here; with enough of them, one will look "significant" by chance. Decide the comparisons you care about *before* looking (the pipeline's list was fixed in advance, and stated in the specification).
* The bootstrap only captures **sampling variability within this world**. It cannot capture the fact that the world might be different next time - that needs repeated seeds (the final report lists this as its main limitation).

## Exercises
1. Rerun Part 1 with `kind = "random_forest"` and a grid you design (`max_depth`, `min_samples_leaf`, `max_features`). Does the edge-of-grid check fire? Widen and repeat.
2. In `paired_block_bootstrap`, resample *rows* instead of *weeks*. How much narrower do the intervals become? Why is that interval misleading?
(Extra: repeat the 8/14/25-week table for horizon 4 and for the Ridge model. Is there a window length at which the conclusion flips?)
3. Compare the ridge model with `alpha = 1` and `alpha = 1000`. Is the difference significant at any horizon?
4. Change the generator seed (`python scripts/01_generate_data.py --seed 7` in a *copy* of the project) and recompute one comparison. Does the ordering of models survive?

**Where this lives in the pipeline:** `src/agriprice/experiment.py` (`tune_model`, `tune_all`), `scripts/03_tune.py`, `src/agriprice/analysis.py` (`paired_block_bootstrap`), `scripts/05_analyse.py` (the comparison list).